# Data Analyst Agent — Privacy Mode

## What is Privacy Mode?

**Privacy Mode** keeps your raw data **on your machine at all times**.

- Only **metadata** (column names, data types, statistics — no actual row values) is sent to OpenAI.
- OpenAI generates Python code based on the metadata.
- That code is **executed locally** on your machine against the real dataframe.
- Charts and output are captured locally and returned to the user.

### When to use Privacy Mode
- When your data contains PII, financial records, or any sensitive information.
- When company policy prohibits sending data to third-party APIs.
- When you want full control over execution.

### Architecture diagram

```
User message + CSV file
        │
        ▼
  FastAPI Backend
        │  reads file locally, generates METADATA only
        │  (no actual data rows are sent)
        ▼
  OpenAI Chat Completions API
  (gpt-4o with metadata context)
        │  returns Python code as text
        ▼
  Backend extracts code from response
        │
        ▼
  execute_code_locally(code, df)
  (code runs on your machine against real df)
        │
        ▼
  stdout + matplotlib charts → returned to frontend
```

## Setup

In [ ]:
# Install required packages (uncomment if needed)
# %pip install openai pandas matplotlib seaborn

In [ ]:
import os
import re
import base64
import traceback
from pathlib import Path
from io import BytesIO

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')  # Non-interactive backend (same as main.py)
import matplotlib.pyplot as plt
import seaborn as sns
import openai
from IPython.display import Image, Markdown, display

OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "sk-...your-key-here...")
client = openai.OpenAI(api_key=OPENAI_API_KEY)

print("Client ready. Data will NOT leave this machine.")

---
## Step 1 — Load Data Locally

The file is read into a pandas DataFrame on your machine. No upload to OpenAI happens.

In `main.py` the dataframe is stored in the session dictionary (lines 953–963 in `/api/upload`).

In [ ]:
DATA_PATH = Path("../backend/sample_data.csv")

df = pd.read_csv(DATA_PATH)
print(f"Loaded locally: {len(df)} rows x {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")
df.head(3)

---
## Step 2 — Generate Metadata (What Gets Sent to OpenAI)

Instead of the raw data, we generate a **metadata string**: column names, types, null counts, unique counts, and statistics for numeric columns. For text columns only the top category names are included — not actual sensitive rows.

This is the `generate_data_metadata()` function in `main.py` (lines 298–322).

In [ ]:
def generate_data_metadata(df) -> str:
    """Reproduce main.py generate_data_metadata() — sends NO raw rows."""
    metadata = []
    metadata.append(f"Dataset: {len(df)} rows x {len(df.columns)} columns")
    metadata.append(f"Memory usage: {df.memory_usage(deep=True).sum() / 1024:.1f} KB")
    metadata.append("\nColumn Information:")

    for col in df.columns:
        dtype = str(df[col].dtype)
        non_null = df[col].notna().sum()
        null_pct = (df[col].isna().sum() / len(df)) * 100
        unique = df[col].nunique()

        col_info = f"  - {col}: {dtype}, {non_null} non-null ({null_pct:.1f}% missing), {unique} unique"

        if pd.api.types.is_numeric_dtype(df[col]):
            stats = df[col].describe()
            col_info += f"\n      Stats: min={stats['min']:.2f}, max={stats['max']:.2f}, mean={stats['mean']:.2f}, std={stats['std']:.2f}"
        elif df[col].dtype == 'object':
            top_values = df[col].value_counts().head(5).index.tolist()
            col_info += f"\n      Top values: {top_values}"  # Only category names, not raw rows

        metadata.append(col_info)

    return "\n".join(metadata)


metadata = generate_data_metadata(df)
print("=== METADATA SENT TO OPENAI ===")
print(metadata)

> Notice: only column names, types, and aggregate statistics are included. No individual car records are sent.

---
## Step 3 — Build the System Prompt and Ask OpenAI for Code

We send the metadata + user question to the **Chat Completions API** (not Assistants). OpenAI is instructed to return Python code wrapped in ` ```python ``` ` markers. No data is attached.

This is lines 1212–1266 inside `chat_privacy_mode()` in `main.py`.

In [ ]:
user_question = "What are the top 5 car brands by average price? Create a bar chart."

system_prompt = f"""You are a data analyst assistant. You help analyze data by writing Python code.

IMPORTANT: You are in PRIVACY MODE. You do NOT have direct access to the actual data values.
You only have this metadata about the dataset:

{metadata}

When the user asks for analysis:
1. Write Python code that performs the requested analysis
2. Assume the dataframe is available as `df`
3. Use print() to output any results or insights
4. Keep code concise and focused

AVAILABLE LIBRARIES (pre-imported): pandas as pd, numpy as np,
matplotlib.pyplot as plt (dark theme pre-configured), seaborn as sns.

ALWAYS wrap your code in ```python and ``` markers.
After the code, provide a brief explanation of what the code does."""

response = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": user_question}
    ],
    max_tokens=2000
)

gpt_response = response.choices[0].message.content
print("=== RAW RESPONSE FROM OPENAI ===")
print(gpt_response)

---
## Step 4 — Extract the Python Code from the Response

We use a regex to pull out any ` ```python ... ``` ` blocks from the model's response. There may be multiple code blocks.

This is lines 1269–1279 in `chat_privacy_mode()` of `main.py`.

In [ ]:
code_pattern = r'```python\n(.*?)```'
code_matches = re.findall(code_pattern, gpt_response, re.DOTALL)

print(f"Found {len(code_matches)} code block(s).")
for i, code in enumerate(code_matches):
    print(f"\n--- Code Block {i+1} ---")
    print(code.strip())

---
## Step 5 — Execute Code Locally Against the Real DataFrame

The extracted code is executed on **your machine** using Python's `exec()`. The real `df` is injected into the execution context, so the code can query actual data without ever sending it to OpenAI.

This is the `execute_code_locally()` function in `main.py` (lines 325–572). Key points:
- A custom `print()` captures stdout instead of printing to console.
- `plt.get_fignums()` captures any matplotlib figures after execution.
- Figures are saved as base64 PNG strings for the frontend.

In [ ]:
def apply_dark_theme():
    """Apply the cyberpunk dark theme used in main.py execute_code_locally()."""
    DARK_BG   = '#0a0e27'
    PLOT_BG   = '#0f1424'
    CYAN      = '#00f0ff'
    PURPLE    = '#7000ff'
    MAGENTA   = '#ff00a0'
    GREEN     = '#10b981'
    ORANGE    = '#f59e0b'
    TEXT      = '#e8edf5'
    TEXT_MUTED= '#a8b4c4'
    GRID      = '#1a2744'
    BORDER    = '#1e3a5f'
    PALETTE   = [CYAN, PURPLE, MAGENTA, GREEN, ORANGE]

    plt.rcdefaults()
    plt.rcParams.update({
        'figure.facecolor': DARK_BG, 'figure.figsize': [10, 6],
        'axes.facecolor': PLOT_BG, 'axes.edgecolor': BORDER,
        'axes.labelcolor': TEXT, 'axes.titlecolor': TEXT,
        'axes.grid': True, 'axes.prop_cycle': plt.cycler(color=PALETTE),
        'grid.color': GRID, 'grid.linestyle': '--', 'grid.alpha': 0.5,
        'xtick.color': TEXT_MUTED, 'ytick.color': TEXT_MUTED,
        'text.color': TEXT,
        'legend.facecolor': '#0c101e', 'legend.labelcolor': TEXT,
        'savefig.facecolor': DARK_BG,
    })
    sns.set_style("darkgrid", {
        'axes.facecolor': PLOT_BG, 'figure.facecolor': DARK_BG,
        'text.color': TEXT, 'xtick.color': TEXT_MUTED, 'ytick.color': TEXT_MUTED,
    })
    sns.set_palette(PALETTE)


def execute_code_locally(code: str, df: pd.DataFrame) -> dict:
    """Reproduce main.py execute_code_locally() — runs code on the local machine."""
    apply_dark_theme()

    output_capture = []
    images = []

    def capture_print(*args, **kwargs):
        output_capture.append(' '.join(str(a) for a in args))

    local_vars = {
        'df': df.copy(),
        'pd': pd, 'np': np,
        'plt': plt, 'sns': sns,
        'BytesIO': BytesIO,
        'print': capture_print,
    }

    try:
        exec(code, {"__builtins__": __builtins__}, local_vars)

        # Capture matplotlib figures
        for fig_num in plt.get_fignums():
            fig = plt.figure(fig_num)
            buf = BytesIO()
            fig.savefig(buf, format='png', dpi=150, bbox_inches='tight',
                        facecolor='#0a0e27', edgecolor='none')
            buf.seek(0)
            images.append(buf.read())
            plt.close(fig)

        return {"success": True, "output": output_capture, "images": images, "error": None}

    except Exception as e:
        plt.close('all')
        return {
            "success": False, "output": output_capture, "images": [],
            "error": f"{type(e).__name__}: {e}\n{traceback.format_exc()}"
        }


print("Helper functions defined.")

In [ ]:
# Execute each code block locally and display results
for i, code in enumerate(code_matches):
    code = code.strip()
    print(f"\n{'='*60}")
    print(f"Executing code block {i+1} LOCALLY (data stays on this machine)")
    print('='*60)
    print(code)
    print()

    result = execute_code_locally(code, df)

    if result["output"]:
        print("--- OUTPUT ---")
        for line in result["output"]:
            print(line)

    for img_bytes in result["images"]:
        print("--- CHART ---")
        display(Image(data=img_bytes))

    if result["error"]:
        print("--- ERROR ---")
        print(result["error"])

---
## Step 6 — Show the Explanation Text

Any text outside the code blocks is the model's explanation of what the code does. In `main.py` this is stripped from the response after extracting code blocks (line 1275).

In [ ]:
# Remove code blocks to get the explanation text
explanation = re.sub(code_pattern, '', gpt_response, flags=re.DOTALL).strip()

if explanation:
    print("=== EXPLANATION FROM OPENAI ===")
    display(Markdown(explanation))
else:
    print("(No explanation text outside code blocks.)")

---
## Step 7 — Multi-turn Conversation in Privacy Mode

Unlike Full Mode, Privacy Mode does **not** maintain a thread on OpenAI. Conversation history must be passed manually in the messages list, or simply start a new round with fresh metadata context.

Here we send a follow-up question:

In [ ]:
follow_up = "Show the distribution of car prices as a histogram."

response2 = client.chat.completions.create(
    model="gpt-4o",
    messages=[
        {"role": "system", "content": system_prompt},
        {"role": "user",   "content": follow_up}
    ],
    max_tokens=2000
)

gpt_response2 = response2.choices[0].message.content
code_matches2 = re.findall(code_pattern, gpt_response2, re.DOTALL)
print(f"Got {len(code_matches2)} code block(s) for follow-up.")

for code in code_matches2:
    result = execute_code_locally(code.strip(), df)
    if result["output"]:
        for line in result["output"]:
            print(line)
    for img_bytes in result["images"]:
        display(Image(data=img_bytes))
    if result["error"]:
        print(result["error"])

---
## Step 8 — Direct Code Execution (No AI)

The agent also exposes a `/api/execute` endpoint that runs user-written code directly — no AI involved at all. This is also entirely local.

In `main.py` this is the `execute_code_endpoint()` handler (lines 1323–1395).

In [ ]:
# Run your own code directly against the dataframe
user_code = """
# Scatter plot: engine size vs price, colored by fuel type
fuel_colors = {'gas': '#00f0ff', 'diesel': '#ff00a0'}

fig, ax = plt.subplots(figsize=(10, 6))
for fuel, group in df.groupby('fuel-type'):
    ax.scatter(group['engine-size'], group['price'],
               c=fuel_colors.get(fuel, '#7000ff'),
               label=fuel, alpha=0.7, s=50)

ax.set_xlabel('Engine Size')
ax.set_ylabel('Price ($)')
ax.set_title('Engine Size vs Price by Fuel Type')
ax.legend()
plt.tight_layout()

print(f"Correlation (engine-size vs price): {df['engine-size'].corr(df['price']):.3f}")
"""

result = execute_code_locally(user_code, df)

if result["output"]:
    for line in result["output"]:
        print(line)

for img_bytes in result["images"]:
    display(Image(data=img_bytes))

if result["error"]:
    print(result["error"])

---
## Summary — What Privacy Mode Does

| Step | What happens | Key `main.py` location |
|------|-------------|------------------------|
| 1 | Load CSV into local DataFrame | `/api/upload`, session `dataframe` key |
| 2 | Generate metadata (no raw rows) | `generate_data_metadata()` |
| 3 | Send metadata + question to Chat Completions API | `chat_privacy_mode()` |
| 4 | Extract ` ```python``` ` code blocks with regex | `chat_privacy_mode()` |
| 5 | Execute each code block locally against real `df` | `execute_code_locally()` |
| 6 | Capture stdout + matplotlib figures as base64 | `execute_code_locally()` |
| 7 | Return explanation text + results to frontend | `chat_privacy_mode()` |
| 8 | Direct code execution (no AI) also available | `/api/execute` |

### Data Flow Comparison

```
FULL MODE                          PRIVACY MODE
──────────────────────────────     ──────────────────────────────
Data rows ──► OpenAI servers       Data rows ──► stays local ✓
Code runs in OpenAI sandbox        Code runs on YOUR machine ✓
Charts downloaded from OpenAI      Charts generated locally ✓
Metadata ──► OpenAI (implicit)     Metadata ──► OpenAI (explicit)
Uses Assistants API + threads      Uses Chat Completions API
```

### Pros and Cons of Privacy Mode

| Pros | Cons |
|------|------|
| Raw data never leaves your machine | Model works with metadata, not real values |
| Works with sensitive / confidential data | Code may fail if metadata is misleading |
| Fast local execution | No persistent thread memory across turns |
| Cheaper (Chat API vs Assistants API) | Model cannot debug by seeing actual errors |

For a mode where OpenAI sees the real data, see **`chat_full_mode.ipynb`**.